# 02 构建交易日流日干支（日柱）

目的：把交易日 `trade_date` 映射为日柱干支字段（`stem/branch/ganzhi_day/jiazi_idx`），并做最小校验。

输入：`data/cache/trade_cal/trade_cal.csv.gz`

输出：`data/clean/ganzhi_trade_dates.csv.gz`（含 `li_chun_year/year_stem/year_element`）

质量检查（落盘到 `data/clean/quality/`）：
- `li_chun_year_mapping_summary.csv`
- `li_chun_year_boundary_sample.csv`


In [1]:
from __future__ import annotations

import datetime as _dt
from pathlib import Path

import numpy as np
import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()

    # Typical: notebook is executed under project root or under ./notebooks
    for candidate in [here] + list(here.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    # Fallback: if executed from a parent folder, look for a child project dir
    for candidate in here.glob('*'):
        if candidate.is_dir() and (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    return here


ROOT = find_project_root()
print('PROJECT_ROOT =', ROOT)


STEMS = list('甲乙丙丁戊己庚辛壬癸')
BRANCHES = list('子丑寅卯辰巳午未申酉戌亥')


def gregorian_to_jdn(date: _dt.date) -> int:
    # Gregorian calendar date -> Julian Day Number (JDN)
    a = (14 - date.month) // 12
    y = date.year + 4800 - a
    m = date.month + 12 * a - 3
    return date.day + (153 * m + 2) // 5 + 365 * y + y // 4 - y // 100 + y // 400 - 32045


def build_jiazi_cycle() -> list[str]:
    return [STEMS[i % 10] + BRANCHES[i % 12] for i in range(60)]


JIAZI = build_jiazi_cycle()


def ganzhi_to_idx(gz: str) -> int:
    try:
        return JIAZI.index(gz)
    except ValueError as exc:
        raise ValueError(f'Invalid ganzhi: {gz!r}') from exc


def date_to_ganzhi(
    date: _dt.date,
    *,
    anchor_date: _dt.date,
    anchor_ganzhi: str,
) -> tuple[int, str, str, str]:
    anchor_jdn = gregorian_to_jdn(anchor_date)
    anchor_idx = ganzhi_to_idx(anchor_ganzhi)

    jdn = gregorian_to_jdn(date)
    jiazi_idx = (jdn - anchor_jdn + anchor_idx) % 60

    gz = JIAZI[jiazi_idx]
    stem = STEMS[jiazi_idx % 10]
    branch = BRANCHES[jiazi_idx % 12]
    return int(jiazi_idx), stem, branch, gz


# =====================
# Anchor（改动必须同步更新校验向量）
# =====================
ANCHOR_DATE = _dt.date(1984, 2, 2)
ANCHOR_GANZHI_DAY = '丙寅'

# 最小校验向量（请后续补充更多日期；来源需来自权威万年历/历书）
TEST_VECTORS: dict[_dt.date, str] = {
    _dt.date(1984, 2, 2): '丙寅',
    _dt.date(2024, 2, 10): '甲辰',
    _dt.date(2026, 2, 13): '戊午',
}

for d, expected in TEST_VECTORS.items():
    _, _, _, got = date_to_ganzhi(d, anchor_date=ANCHOR_DATE, anchor_ganzhi=ANCHOR_GANZHI_DAY)
    assert got == expected, (d, got, expected)

print('Test vectors OK:', {d.isoformat(): v for d, v in TEST_VECTORS.items()})

PROJECT_ROOT = D:\Work\中国传统投资\风水五行阴阳天干地支
Test vectors OK: {'1984-02-02': '丙寅', '2024-02-10': '甲辰', '2026-02-13': '戊午'}


In [2]:
# 读取交易日并生成干支表（含立春年 year_element）
trade_cal_path = ROOT / 'data/cache/trade_cal/trade_cal.csv.gz'
if not trade_cal_path.exists():
    raise FileNotFoundError('Missing cache: data/cache/trade_cal/trade_cal.csv.gz. Run notebook 01 first.')

cal = pd.read_csv(trade_cal_path, compression='gzip', dtype={'cal_date': str})
trade_dates = (
    cal.loc[cal['is_open'] == 1, ['cal_date']]
    .rename(columns={'cal_date': 'trade_date'})
    .drop_duplicates()
    .sort_values(['trade_date'])
    .reset_index(drop=True)
)

li_chun_map_path = ROOT / 'docs/li_chun_year_mapping.csv'
if not li_chun_map_path.exists():
    raise FileNotFoundError('Missing docs/li_chun_year_mapping.csv. Please update project docs.')

li_chun_map = pd.read_csv(
    li_chun_map_path,
    dtype={
        'li_chun_year': int,
        'li_chun_date_yyyymmdd': str,
        'li_chun_begin_local': str,
        'year_stem': str,
        'year_element': str,
    },
)
required_cols = {'li_chun_year', 'li_chun_date_yyyymmdd', 'li_chun_begin_local', 'year_stem', 'year_element'}
missing_cols = required_cols - set(li_chun_map.columns)
if missing_cols:
    raise ValueError(f'li_chun_year_mapping.csv missing columns: {sorted(missing_cols)}')

li_chun_map = (
    li_chun_map.drop_duplicates(subset=['li_chun_year'])
    .sort_values(['li_chun_year'])
    .reset_index(drop=True)
)
li_chun_map['li_chun_date_yyyymmdd'] = (
    li_chun_map['li_chun_date_yyyymmdd']
    .astype(str)
    .str.replace('.0', '', regex=False)
    .str.zfill(8)
)
li_chun_map['li_chun_time_hm'] = li_chun_map['li_chun_begin_local'].astype(str).str[-5:]

li_chun_date_by_year = li_chun_map.set_index('li_chun_year')['li_chun_date_yyyymmdd'].to_dict()
li_chun_time_by_year = li_chun_map.set_index('li_chun_year')['li_chun_time_hm'].to_dict()
year_stem_by_year = li_chun_map.set_index('li_chun_year')['year_stem'].to_dict()
year_element_by_year = li_chun_map.set_index('li_chun_year')['year_element'].to_dict()


def _to_date(yyyymmdd: str) -> _dt.date:
    return _dt.date(int(yyyymmdd[0:4]), int(yyyymmdd[4:6]), int(yyyymmdd[6:8]))


records = []
for s in trade_dates['trade_date'].tolist():
    d = _to_date(s)
    jiazi_idx, stem, branch, gz = date_to_ganzhi(d, anchor_date=ANCHOR_DATE, anchor_ganzhi=ANCHOR_GANZHI_DAY)

    trade_year = int(s[0:4])
    li_chun_date = li_chun_date_by_year.get(trade_year)
    if li_chun_date is None:
        raise ValueError(f'Missing li_chun_date for trade_year={trade_year} in docs/li_chun_year_mapping.csv')

    li_chun_time = li_chun_time_by_year.get(trade_year)
    if li_chun_time is None:
        raise ValueError(f'Missing li_chun_begin_local for trade_year={trade_year} in docs/li_chun_year_mapping.csv')

    if s < li_chun_date:
        li_chun_year = trade_year - 1
    elif s > li_chun_date:
        li_chun_year = trade_year
    else:
        # trade_date == li_chun_date: decide by market close time (15:00) vs li_chun moment
        li_chun_year = trade_year if li_chun_time <= '15:00' else trade_year - 1
    year_stem = year_stem_by_year.get(li_chun_year)
    year_element = year_element_by_year.get(li_chun_year)
    if (year_stem is None) or (year_element is None):
        raise ValueError(f'Missing year mapping for li_chun_year={li_chun_year} in docs/li_chun_year_mapping.csv')
    records.append(
        {
            'trade_date': s,
            'jiazi_idx': jiazi_idx,
            'stem': stem,
            'branch': branch,
            'ganzhi_day': gz,
            'li_chun_year': int(li_chun_year),
            'year_stem': str(year_stem),
            'year_element': str(year_element),
        }
    )

out = pd.DataFrame.from_records(records)

# 质量检查：立春年映射覆盖与边界样本
quality_dir = ROOT / 'data/clean/quality'
quality_dir.mkdir(parents=True, exist_ok=True)

if out[['li_chun_year', 'year_stem', 'year_element']].isna().any().any():
    raise ValueError('Found missing li_chun_year/year_stem/year_element; please check docs/li_chun_year_mapping.csv')

summary = (
    out.groupby(['li_chun_year', 'year_stem', 'year_element'], dropna=False)
    .agg(
        n_trade_dates=('trade_date', 'size'),
        min_trade_date=('trade_date', 'min'),
        max_trade_date=('trade_date', 'max'),
    )
    .reset_index()
    .sort_values(['li_chun_year'])
)

summary_path = quality_dir / 'li_chun_year_mapping_summary.csv'
summary.to_csv(summary_path, index=False)
print('Saved:', summary_path)

tmp = out.copy()
tmp['date'] = pd.to_datetime(tmp['trade_date'], format='%Y%m%d')
tmp['trade_year'] = tmp['date'].dt.year.astype(int)

boundary_parts = []
for y in sorted(tmp['trade_year'].unique().tolist()):
    li_chun_date = li_chun_date_by_year.get(int(y))
    if li_chun_date is None:
        continue
    li_chun_time = li_chun_time_by_year.get(int(y))
    if li_chun_time is None:
        continue
    lc = pd.to_datetime(li_chun_date, format='%Y%m%d')
    mask = (tmp['date'] >= lc - pd.Timedelta(days=3)) & (tmp['date'] <= lc + pd.Timedelta(days=3))
    sub = tmp.loc[mask, ['trade_date', 'li_chun_year', 'year_stem', 'year_element']].copy()
    if sub.empty:
        continue
    sub.insert(0, 'li_chun_date_yyyymmdd', li_chun_date)
    sub.insert(1, 'li_chun_time_hm', li_chun_time)
    sub.insert(2, 'market_close_hm', '15:00')
    sub.insert(0, 'trade_year', int(y))
    sub['is_before_li_chun_date'] = (sub['trade_date'] < li_chun_date).astype(int)
    sub['li_chun_after_close'] = int(li_chun_time > '15:00')
    boundary_parts.append(sub)

boundary = pd.concat(boundary_parts, ignore_index=True) if boundary_parts else pd.DataFrame()
boundary_path = quality_dir / 'li_chun_year_boundary_sample.csv'
boundary.to_csv(boundary_path, index=False)
print('Saved:', boundary_path)
out_path = ROOT / 'data/clean/ganzhi_trade_dates.csv.gz'
out_path.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(out_path, index=False, compression='gzip')

print('Saved:', out_path)
out.head()


Saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean\quality\li_chun_year_mapping_summary.csv
Saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean\quality\li_chun_year_boundary_sample.csv
Saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean\ganzhi_trade_dates.csv.gz


,trade_date,jiazi_idx,stem,branch,ganzhi_day,li_chun_year,year_stem,year_element
0,20100104,50,甲,寅,甲寅,2009,己,土
1,20100105,51,乙,卯,乙卯,2009,己,土
2,20100106,52,丙,辰,丙辰,2009,己,土
3,20100107,53,丁,巳,丁巳,2009,己,土
4,20100108,54,戊,午,戊午,2009,己,土
